# Atelier 4.A — Agent RAG multi-sources ★

**Durée : 90 min** · Projet fil rouge, incrément du Jour 4

L'assistant du Jour 2 connaît la documentation mais ne fait qu'une recherche ; l'agent du Jour 3 consulte le CRM mais ignore la documentation. L'**agent RAG** réunit les deux : la recherche devient un **outil** parmi d'autres, et l'agent décide quelles sources interroger, dans quel ordre, et combien de fois.

Il répond au client connecté au portail de Fictiva à partir de trois sources :

| Source | Contenu | Outils | Référence citée |
|---|---|---|---|
| Documentation (Jour 2) | CGV, retours, livraison, maintenance, catalogue, FAQ, portail | `rechercher_documentation` | `[DOC-12]` |
| Base de connaissances du support | pannes, codes d'erreur, procédures de dépannage | `rechercher_support` | `[KB-104]` |
| CRM (Jour 3) | fiche du client, commandes, calcul de remboursement | `fiche_client`, `mes_commandes`, `consulter_commande`, `calculer_remboursement` | `[CMD-2026-0412]` |

Livrable : l'agent, exposé par le service web du Jour 2 (`POST /agent`). Les briques sont dans `agent_rag.py`.

In [1]:
import json
import subprocess
import sys
import time

import httpx
import pandas as pd

import agent_rag as ar
import outils_llm as llm
import rag

MODELE = llm.choisir_modele()   # le modèle doit savoir appeler des outils
print("Modèle :", MODELE)

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Modèle : groq:openai/gpt-oss-20b


## 1. Les sources de connaissance

La documentation est découpée et indexée comme au Jour 2 (sections, en-têtes, recherche hybride et reranker). La base du support est un **second index** : un article par extrait, avec ses produits. Deux index séparés, c'est le **RAG multi-index** : chaque source garde son vocabulaire, et l'agent choisit où chercher.

In [2]:
sources = ar.construire_sources()   # premier lancement : téléchargement des modèles (0,8 Go)
print(len(sources.documentation.extraits), "extraits de documentation,", len(sources.support.extraits), "articles de support")
pd.DataFrame(ar.charger_articles())[["id", "titre", "produits", "confidentialite"]]

67 extraits de documentation, 21 articles de support


,id,titre,produits,confidentialite
0,KB-101,LX-500 : bourrages papier répétés,"[LX-500, PA-4R]",public
1,KB-102,LX-500 : l'imprimante n'apparaît pas sur le ré...,[LX-500],public
2,KB-103,LX-500 : impression pâle ou traces sur les pages,"[LX-500, TN-500]",public
3,KB-104,LX-500 : voyant rouge clignotant et code d'err...,[LX-500],public
4,KB-201,EC-27Q : pas d'image avec un ordinateur raccor...,"[EC-27Q, NB-14]",public
5,KB-202,DK-200 : les deux écrans affichent la même ima...,"[DK-200, EC-27Q]",public
6,KB-203,DK-200 : l'ordinateur portable ne charge pas s...,"[DK-200, NB-14]",public
7,KB-301,KB-80 et MS-30 : clavier ou souris sans fil qu...,"[KB-80, MS-30]",public
8,KB-302,KB-80 : les lettres Y et Z sont inversées ou l...,[KB-80],public
9,KB-401,NB-14 : la batterie ne se charge pas au-delà d...,[NB-14],public


L'article `KB-900` est une **note interne** : comme la procédure interne du Jour 2, il est filtré **avant** la recherche (`public_seulement=True`). L'agent ne peut pas citer ce qu'il ne reçoit pas.

In [3]:
for source, requete in [("documentation", "délai de retour d'un article sans défaut"),
                        ("support", "LX-500 code E-21 voyant rouge"),
                        ("support", "défaut connu sur un lot de LX-500")]:
    print(f"== {source} : {requete}")
    for e in sources.rechercher(source, requete):
        print(f"   [{ar.reference(e)}] {e.reference()[:80]}  (score {e.score:.2f})")

== documentation : délai de retour d'un article sans défaut
   [DOC-9] Politique de retours et de remboursement, Fictiva SA — Politique de retours et d  (score 3.66)
   [DOC-7] Politique de retours et de remboursement, Fictiva SA — Politique de retours et d  (score 2.17)
   [DOC-8] Politique de retours et de remboursement, Fictiva SA — Politique de retours et d  (score 1.37)
   [DOC-10] Politique de retours et de remboursement, Fictiva SA — Politique de retours et d  (score -1.42)
== support : LX-500 code E-21 voyant rouge
   [KB-104] LX-500 : voyant rouge clignotant et code d'erreur E-21 (KB-104)  (score 6.87)
   [KB-102] LX-500 : l'imprimante n'apparaît pas sur le réseau Wi-Fi (KB-102)  (score -3.10)
   [KB-103] LX-500 : impression pâle ou traces sur les pages (KB-103)  (score -3.31)
== support : défaut connu sur un lot de LX-500
   [KB-103] LX-500 : impression pâle ou traces sur les pages (KB-103)  (score 0.50)
   [KB-101] LX-500 : bourrages papier répétés (KB-101)  (score 0.33)
   

## 2. Le RAG naïf face à une question multi-sources

Une cliente de Rochat Emballages écrit : *« Une des imprimantes LX-500 que nous venons de recevoir affiche le code E-21. Est-elle sous garantie et que devons-nous faire ? »*

Il faut trois informations : la date de livraison (CRM), la durée de la garantie (CGV) et la signification du code E-21 (support). L'assistant du Jour 2 fait **une** recherche, dans **une** source :

In [4]:
QUESTION_E21 = ("Une des imprimantes LX-500 que nous venons de recevoir affiche le code E-21. "\
                "Est-elle sous garantie et que devons-nous faire ?")
assistant_j2 = rag.Assistant(sources.documentation, sources.reranker, modele=MODELE)
r = assistant_j2.repondre(QUESTION_E21)
print(r.texte, "\n\nSources :", r.sources())

Je ne trouve pas cette information dans la documentation de Fictiva. 

Sources : []


Au mieux une réponse partielle : la documentation ne connaît ni la commande du client, ni le code E-21.

## 3. Les outils de l'agent

Chaque outil renvoie des extraits ou des fiches **précédés de leur référence**, que l'agent doit citer. Les outils du CRM ne prennent **pas** le numéro de client en paramètre : il vient de la session du portail, transmis dans la configuration de l'exécution (`RunnableConfig`). Le modèle ne le voit pas et ne peut pas le changer : c'est le **moindre privilège** appliqué à l'identité.

In [5]:
outils = ar.creer_outils(sources)
for o in outils:
    print(f"{o.name}({', '.join(o.args)})")

rechercher_documentation(question)
rechercher_support(question)
fiche_client()
mes_commandes()
consulter_commande(numero_commande)
calculer_remboursement(numero_commande, references, defectueux, consommables_ouverts)


In [6]:
consulter = next(o for o in outils if o.name == "consulter_commande")
print(consulter.invoke({"numero_commande": "CMD-2026-0412"}, config=ar.config("CL-10234"))[:300], "\n")
# la même commande, demandée depuis le compte d'un autre client :
print(consulter.invoke({"numero_commande": "CMD-2026-0412"}, config=ar.config("CL-10456")))

[CMD-2026-0412] {"numero_commande": "CMD-2026-0412", "client": {"numero_client": "CL-10234", "nom": "Rochat Emballages SA", "statut": "standard"}, "statut": "livree", "date_commande": "2026-10-16", "date_livraison": "2026-10-19", "lignes": [{"reference": "LX-500", "quantite": 3, "prix_unitaire": 400 

Commande CMD-2026-0412 introuvable dans le compte du client connecté.


Le message d'erreur ne dit pas si la commande existe ailleurs : il ne révèle rien à un curieux.

## 4. L'agent

Même graphe qu'au Jour 3 (`agent ↔ outils`, avec mémoire), mais d'autres outils et un autre prompt. Le prompt donne la **méthode** : repérer les sources nécessaires, ne rien affirmer sans outil, citer les références, refuser hors du domaine.

In [7]:
print(ar.PROMPT_AGENT)
agent = ar.construire_agent(outils, modele=MODELE)
print(agent.get_graph().draw_mermaid())

Tu es l'assistant du portail client de Fictiva SA, distributeur B2B de matériel informatique.
Tu réponds au client connecté : {client}. Nous sommes le 23 octobre 2026.

Tes sources, à interroger avec les outils :
- rechercher_documentation : conditions générales, garantie, retours, livraison, maintenance, catalogue, FAQ, portail ;
- rechercher_support : pannes, codes d'erreur et procédures de dépannage des produits ;
- fiche_client, mes_commandes, consulter_commande, calculer_remboursement : le compte du client connecté.

Méthode :
1. Repère les informations nécessaires et la source de chacune : une question peut en demander plusieurs.
2. N'affirme rien qui ne vienne d'un résultat d'outil. Pour une garantie ou un retour, pars de la date de livraison (consulter_commande donne aussi le nombre de jours écoulés). Calcule un remboursement avec calculer_remboursement ; pour tout autre calcul, détaille-le à partir des chiffres trouvés.
3. Cite chaque information avec la référence donnée par l

Cinq questions de nature différente : documentation seule, support seul, CRM seul, plusieurs sources, hors du domaine. Observez les **trajectoires** : quels outils, dans quel ordre, combien d'appels au modèle.

In [8]:
QUESTIONS = [
    ("CL-10456", "Livrez-vous au Liechtenstein, et en combien de temps ?"),
    ("CL-10456", "Quand je tape un Z sur mon clavier KB-80, c'est un Y qui s'affiche. Que faire ?"),
    ("CL-10234", "Quand ma dernière commande a-t-elle été livrée ?"),
    ("CL-10234", QUESTION_E21),
    ("CL-10234", "Quel temps fera-t-il demain à Lausanne ?"),
]
resultats_agent = {}
for client, question in QUESTIONS:
    print("=" * 100, f"\n{client} : {question}\n")
    resultats_agent[question] = r = await ar.demander(agent, client, question, f"atelier-{len(resultats_agent)}")
    ar.afficher(r, sources)

CL-10456 : Livrez-vous au Liechtenstein, et en combien de temps ?

🔧 rechercher_documentation({"question": "livraison Liechtenstein délai"})

🤖 Bonjour,  

Oui, nous livrons au Liechtenstein. Le délai est de **trois jours ouvrables** après confirmation de votre commande [DOC-36].  

Les frais d’expédition sont de **45 CHF** si la valeur hors taxes est inférieure à 500 CHF, sinon la livraison est gratuite [DOC-36].  

Si vous avez d’autres questions sur votre commande, n’hésitez pas à me le faire savoir.  

Cordialement,  
L’équipe Fictiva SA

   [DOC-36] Guide de livraison, Fictiva SA — Guide de livraison > Zones et délais (guide_livraison.md)

   2 appels au modèle, 1 appels d'outils, 3015 tokens, 3.5 s
CL-10456 : Quand je tape un Z sur mon clavier KB-80, c'est un Y qui s'affiche. Que faire ?

🔧 rechercher_support({"question": "KB-80 clavier Z affiche Y, problème de configuration ou dysfonctionnement?"})

🤖 Vous avez rencontré un problème de disposition clavier.  
[KB-302] indique que

In [9]:
pd.DataFrame([{"question": q[:60], "sources": ", ".join(r.sources_utilisees()) or "aucune",
               "appels_modele": r.appels_modele, "appels_outils": len(r.outils), "tokens": r.tokens,
               "latence_s": r.latence_s} for q, r in resultats_agent.items()])

,question,sources,appels_modele,appels_outils,tokens,latence_s
0,"Livrez-vous au Liechtenstein, et en combien de...",documentation,2,1,3015,3.5
1,"Quand je tape un Z sur mon clavier KB-80, c'es...",support,2,1,2807,3.7
2,Quand ma dernière commande a-t-elle été livrée ?,crm,2,1,2186,2.0
3,Une des imprimantes LX-500 que nous venons de ...,"crm, support",3,2,4617,20.4
4,Quel temps fera-t-il demain à Lausanne ?,aucune,1,0,1040,13.0


**À observer.** Pour la météo, l'agent n'appelle aucun outil. Pour la question sur le code E-21, il devrait interroger les trois sources : vérifiez dans sa trajectoire qu'il a bien consulté la date de livraison. S'il s'en est passé, sa réponse sur la garantie reste générale (« garanti 24 mois ») au lieu de dire si **cette** imprimante est couverte : c'est une erreur de trajectoire, que l'atelier 4.C mesure (rappel des sources). Le nombre d'appels au modèle varie avec la difficulté : c'est la force de l'agent (il s'adapte) et son coût (latence et tokens).

## 5. Des citations vérifiables

Une citation n'a de valeur que si elle renvoie à un extrait réellement transmis à l'agent. On compare les références **citées** dans la réponse et les références **renvoyées** par les outils : une citation qui ne correspond à aucun résultat d'outil est inventée.

In [10]:
r = resultats_agent[QUESTION_E21]
print("Renvoyées par les outils :", r.renvoyees)
print("Citées dans la réponse   :", r.citees)
print("Citations inventées      :", r.invalides or "aucune")

Renvoyées par les outils : ['CMD-2026-0412', 'KB-104', 'KB-102', 'KB-101']
Citées dans la réponse   : ['CMD-2026-0412', 'KB-104']
Citations inventées      : aucune


In [11]:
# Une réponse fabriquée pour l'exemple : [DOC-58] n'a été renvoyé par aucun outil.
fausse = "Votre imprimante est garantie 36 mois [DOC-58] et le code E-21 indique un bourrage [KB-104]."
print("Citations inventées :", [ref for ref in ar.references_citees(fausse) if ref not in r.renvoyees])

Citations inventées : ['DOC-58']


Le contrôle repère une **référence inventée**, pas une **affirmation fausse** attribuée à une vraie référence (« bourrage » avec [KB-104]) : pour cela, il faut un juge de fidélité, que l'on mettra en place dans l'atelier 4.C.

## 6. Une conversation sur plusieurs tours

La mémoire (*checkpointer*) conserve les messages de chaque conversation (`thread_id`). L'agent peut répondre à « ces claviers » sans que la cliente répète le numéro de commande. Le fil de conversation est préfixé par le numéro de client : un client ne peut pas reprendre la conversation d'un autre.

In [12]:
TOURS = ["Quelle est ma dernière commande ?",
         "Si je retourne ces claviers sans défaut, combien serai-je remboursée ?",
         "Et qui paie l'envoi du retour ?"]
for message in TOURS:
    print("=" * 100, f"\n👤 {message}\n")
    ar.afficher(await ar.demander(agent, "CL-10456", message, "conversation-claviers"), sources)

👤 Quelle est ma dernière commande ?

🔧 mes_commandes({"question": "?"})

🤖 Votre dernière commande porte le numéro : **CMD-2026-0415** [CMD-2026-0415].  
Elle a été passée le 07 octobre 2026 et livrée le 09 octobre 2026, avec le statut « livrée » [CMD-2026-0415].  
Les références de produits commandées sont : KB-80 [CMD-2026-0415].  
Si vous avez besoin de plus d’informations (facture, suivi, etc.), n’hésitez pas à consulter votre espace client.

   [CMD-2026-0415] Commande CMD-2026-0415 (CRM)

   2 appels au modèle, 1 appels d'outils, 2309 tokens, 1.4 s
👤 Si je retourne ces claviers sans défaut, combien serai-je remboursée ?

🔧 consulter_commande({"numero_commande": "CMD-2026-0415"})
🔧 calculer_remboursement({"defectueux": false, "numero_commande": "CMD-2026-0415", "references": ["KB-80"]})

🤖 Vous pouvez retourner vos trois claviers sans défaut : le montant remboursable sera de **216 CHF** [CMD-2026-0415].  
La politique de retour permet le remboursement intégral dans les 30 jours su

In [13]:
etat = await agent.aget_state(ar.config("CL-10456", "conversation-claviers"))
print(len(etat.values["messages"]), "messages en mémoire dans cette conversation")
# Une nouvelle conversation repart de zéro :
ar.afficher(await ar.demander(agent, "CL-10456", "Et qui paie l'envoi du retour ?", "nouvelle-conversation"))

14 messages en mémoire dans cette conversation
🔧 rechercher_documentation({"question": "qui paie l'envoi du retour Fictiva"})

🤖 Bonjour ; selon notre FAQ, les frais d’envoi d’un retour sont à la charge du client sauf si l’article est défectueux ou si l’erreur provient de Fictiva ; dans ces cas, l’étiquette de retour est fournie gratuitement [DOC-24]. Pour un retour sans défaut, le client doit s’acquitter du coût d’expédition initial et vous ne pourrez pas récupérer ces frais [DOC-9]. En revanche, si l’article est défectueux et couvert par la garantie de 24 mois, Fictiva remboursera intégralement le prix de l’article et les frais d’expédition initiaux [DOC-8]. N’hésitez pas à nous contacter si vous avez besoin d’une étiquette prépayée ou d’autres précisions.


   2 appels au modèle, 1 appels d'outils, 3001 tokens, 12.1 s


Le contexte grandit à chaque tour : sur une longue conversation, on résume les anciens tours ou on ne garde que les derniers messages.

## 7. Routeur explicite ou agent ? (Adaptive RAG)

Un agent coûte plusieurs appels au modèle, même pour une question simple. L'**Adaptive RAG** place un **routeur** en amont : un appel de classification choisit la stratégie la moins coûteuse.

| Stratégie | Traitement | Appels au modèle |
|---|---|---|
| `documentation` ou `support` | une recherche, une génération citée (RAG simple) | 2 (routeur + génération) |
| `agent` | l'agent multi-sources | 1 + ceux de l'agent |
| `hors_perimetre` | réponse fixe, sans recherche | 1 |

In [14]:
async def repondre_adaptatif(client: str, question: str) -> dict:
    debut = time.perf_counter()
    routage = ar.router(question, MODELE)
    appels = 1
    if routage.strategie in ("documentation", "support"):
        extraits = sources.rechercher(routage.strategie, question)
        texte, appels = ar.generer(question, extraits, MODELE).texte, appels + 1
    elif routage.strategie == "hors_perimetre":
        texte = ar.REPONSE_HORS_PERIMETRE
    else:
        r = await ar.demander(agent, client, question, f"adaptatif-{time.time()}")
        texte, appels = r.reponse, appels + r.appels_modele
    return {"strategie": routage.strategie, "justification": routage.justification, "reponse": texte,
            "appels_modele": appels, "latence_s": round(time.perf_counter() - debut, 1)}

comparaison = []
for client, question in QUESTIONS:
    adaptatif = await repondre_adaptatif(client, question)
    seul = resultats_agent[question]
    comparaison.append({"question": question[:55], "strategie": adaptatif["strategie"],
                        "appels (routeur)": adaptatif["appels_modele"], "appels (agent seul)": seul.appels_modele,
                        "latence (routeur)": adaptatif["latence_s"], "latence (agent seul)": seul.latence_s})
    print(f"{adaptatif['strategie']:>15} | {question[:55]}\n                  {adaptatif['reponse'][:160]}\n")
pd.DataFrame(comparaison)

  documentation | Livrez-vous au Liechtenstein, et en combien de temps ?
                  Oui, Fictiva SA livre au Liechtenstein. Le délai de livraison standard pour le Liechtenstein est de **3 jours ouvrables** [DOC-36]. Le forfait de livraison s’él

        support | Quand je tape un Z sur mon clavier KB-80, c'est un Y qu
                  Le problème vient de la disposition de clavier que votre système a sélectionnée : il n’est pas configuré sur la disposition suisse romande du KB‑80.  
Sous Wind

          agent | Quand ma dernière commande a-t-elle été livrée ?
                  Bonjour, votre dernière commande, référencée CMD-2026-0412, a été livrée le 19 octobre 2026. [CMD-2026-0412] Si vous avez besoin de plus d’informations, n’hésit

        support | Une des imprimantes LX-500 que nous venons de recevoir 
                  Oui, votre LX‑500 est encore sous garantie de 24 mois.  
Éteignez l’imprimante, débranchez‑la et laissez‑la refroidir 30 minutes, puis rallumez‑la.  
Si l

,question,strategie,appels (routeur),appels (agent seul),latence (routeur),latence (agent seul)
0,"Livrez-vous au Liechtenstein, et en combien de...",documentation,2,2,3.3,3.5
1,"Quand je tape un Z sur mon clavier KB-80, c'es...",support,2,2,5.6,3.7
2,Quand ma dernière commande a-t-elle été livrée ?,agent,3,2,22.5,2.0
3,Une des imprimantes LX-500 que nous venons de ...,support,2,3,12.8,20.4
4,Quel temps fera-t-il demain à Lausanne ?,hors_perimetre,1,1,5.4,13.0


**À discuter.** Le routeur économise des appels sur les questions simples, mais une **erreur de routage est définitive** : une question classée « documentation » qui demandait le CRM obtiendra une réponse incomplète. L'agent seul est plus robuste, mais plus cher. On choisit en mesurant les deux sur le golden set (atelier 4.C).

## 8. Exposer l'agent dans l'API du Jour 2

Le service web du Jour 2 (`api_assistant.py`) garde sa route `POST /question` et gagne `POST /agent`. La requête porte le numéro du client connecté (en production, il viendrait du jeton d'authentification du portail) et, pour poursuivre une conversation, l'identifiant renvoyé par la réponse précédente. Le service applique en plus les garde-fous de l'atelier 4.C.

In [15]:
serveur = subprocess.Popen([sys.executable, "-m", "uvicorn", "api_assistant:app", "--port", "8002"],
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(120):   # le service construit ses index au démarrage
    try:
        print(httpx.get("http://localhost:8002/sante", timeout=2).json())
        break
    except httpx.HTTPError:
        time.sleep(1)

{'statut': 'ok', 'extraits_indexes': 67, 'articles_support': 21, 'modele': 'groq:openai/gpt-oss-20b'}


In [16]:
reponse = httpx.post("http://localhost:8002/agent", timeout=600, json={
    "numero_client": "CL-10901",
    "message": "Notre NAS NS-4 affiche un voyant orange. Est-il encore sous garantie ?"}).json()
print(json.dumps(reponse, ensure_ascii=False, indent=2))

{
  "reponse": "Bonjour, votre NAS NS-4 affiche un voyant orange, signe d’un disque défaillant ou d’une grappe RAID dégradée [DOC-44].  \nLa garantie standard de Fictiva couvre 24 mois à partir de la date de livraison, conformément aux CGV [DOC-3].  \nVotre commande CMD-2023-0150 a été livrée le 12 juin 2023 [CMD-2023-0150], donc la garantie est déjà expirée depuis le 12 juin 2025.  \nJe vous conseille donc de contacter le support technique pour une intervention ou un remplacement éventuel.  \nVous trouverez le numéro de série sur l’appareil pour faciliter la prise en charge.  \nJe reste à votre disposition pour toute autre question.",
  "conversation": "29ed2de5e132",
  "citations": [
    {
      "reference": "DOC-44",
      "source": "Serveur NAS NS-4 — Guide de démarrage rapide (guide_nas_ns4.pdf)"
    },
    {
      "reference": "DOC-3",
      "source": "Conditions générales de vente (cgv_fictiva.pdf)"
    },
    {
      "reference": "CMD-2023-0150",
      "source": "Commande CMD-2

In [17]:
suite = httpx.post("http://localhost:8002/agent", timeout=600, json={
    "numero_client": "CL-10901", "conversation": reponse["conversation"],
    "message": "Combien coûterait un contrat de maintenance Or pour ce serveur ?"}).json()
print(suite["reponse"], "\n\nCitations :", [c["reference"] for c in suite["citations"]], "| outils :", suite["outils"])

Le contrat de maintenance Or est facturé à 18 % de la valeur du matériel ; pour un NAS NS-4 à 1 250 CHF HT, ce serait 225 CHF HT [DOC-42, DOC-34].  
Cependant, la politique tarifaire impose un minimum de 600 CHF HT annuel, quel que soit le niveau choisi [DOC-42].  
Ainsi, le prix annuel du niveau Or pour votre NAS sera de 600 CHF HT.  
N’hésitez pas à nous contacter pour souscrire ou pour toute autre question. 

Citations : ['DOC-42', 'DOC-34'] | outils : ['rechercher_documentation']


In [18]:
serveur.terminate()

✏️ Ouvrez http://localhost:8002/docs pendant que le service tourne : la route `/agent` y est documentée avec ses schémas d'entrée et de sortie.

## À retenir

- Dans un agent RAG, la **recherche est un outil** : l'agent choisit les sources, reformule, recherche à nouveau si besoin.
- Plusieurs **index séparés** (documentation, support) et des données structurées (CRM) : chaque source garde ses filtres d'accès.
- L'identité du client passe par la **configuration**, jamais par le modèle.
- Des **références citables** dans chaque résultat d'outil rendent les citations vérifiables.
- Un **routeur** (Adaptive RAG) réduit le coût des questions simples, au prix d'erreurs de routage définitives.

## Pour aller plus loin

- Ajoutez un outil `rechercher_tickets` sur un historique de tickets résolus (une quatrième source).
- Remplacez les outils CRM locaux par le serveur MCP du Jour 3, en passant l'identité du client par l'authentification du serveur.